# AgentGuard — Parallel Optuna Sweep + Comparison Evaluation

Runs (optionally) the full 4-phase Optuna sweep with **multi-GPU parallelism**, loads best hyperparameters, then trains **two AgentGuard variants** (baseline vs tuned) under identical 5-fold stratified CV. Reports F1, AUROC, AUPRC, precision, recall, accuracy, and per-fold confusion matrices.

**Parallelism strategy:**
- **Phases 1–3**: N_GPUS worker processes share `sweep/results/optuna.db` via `load_if_exists=True`. Each worker pinned to one GPU via `CUDA_VISIBLE_DEVICES`. Optuna auto-coordinates trial distribution.
- **Phase 4**: 25 (config, fold) jobs dispatched via a GPU pool (up to N_GPUS concurrent).
- **Notebook CV**: 10 (variant, fold) jobs dispatched the same way.
- **DataLoader**: `num_workers=4, pin_memory=True, persistent_workers=True` already baked into `sweep/objective.py` and `scripts/run_fold.py`.
- **Storage**: default SQLite with WAL mode. For >7 workers, set `OPTUNA_STORAGE=postgresql://...` env var.

**Running on the rented GPU box:**
```bash
tmux new -s notebook
jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --NotebookApp.token=''
# From laptop: ssh -L 8888:localhost:8888 user@gpu-host
```

In [ ]:
# Cell 1 — Flags + imports
import concurrent.futures
import copy
import json
import math
import os
import queue
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

from sweep.run_sweep import load_base_config
from sweep.config_override import override_config
from main import make_stratified_folds

# --- Flip this to True on the GPU box to actually run the sweep ---
RUN_SWEEP = False

# --- Parallelism: set to however many GPUs you rented ---
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1

PHASE_TRIALS = {1: 100, 2: 60, 3: 60}
PHASE4_FOLDS = 5
PHASE4_TOP_K = 5
N_FOLDS = 5

RESULTS_DIR = REPO_ROOT / "results"
SWEEP_RESULTS = REPO_ROOT / "sweep" / "results"
LOG_DIR = REPO_ROOT / "logs" / "parallel"
for d in (RESULTS_DIR, SWEEP_RESULTS, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Repo root:      {REPO_ROOT}")
print(f"RUN_SWEEP:      {RUN_SWEEP}")
print(f"N_GPUS:         {N_GPUS}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"OPTUNA_STORAGE: {os.environ.get('OPTUNA_STORAGE') or 'sqlite (default)'}")

In [ ]:
# Cell 2 — Base config + SQLite WAL setup
base_config = load_base_config(str(REPO_ROOT / "config.yml"))

data_cfg = base_config["data"]
print(f"Attacked agents ({len(data_cfg['attacked_agents'])}): {data_cfg['attacked_agents']}")
print(f"Control agents  ({len(data_cfg['control_agents'])}): {data_cfg['control_agents']}")
print(f"k_folds: {data_cfg['k_folds']}  seq_context: {data_cfg['seq_context']}")
print(f"batch_size: {base_config['training']['batch_size']}  epochs: {base_config['training']['epochs']}")

# Enable WAL mode on the SQLite Optuna DB so multiple workers can write concurrently.
# Idempotent — safe to run every time.
optuna_db = SWEEP_RESULTS / "optuna.db"
if not os.environ.get("OPTUNA_STORAGE"):
    optuna_db.touch(exist_ok=True)
    try:
        subprocess.run(
            ["sqlite3", str(optuna_db), "PRAGMA journal_mode=WAL;"],
            check=True, capture_output=True, text=True,
        )
        print(f"SQLite WAL mode enabled on {optuna_db}")
    except (FileNotFoundError, subprocess.CalledProcessError) as e:
        print(f"WARN: couldn't enable WAL via sqlite3 CLI ({e}). "
              f"Run manually on GPU box: sqlite3 {optuna_db} 'PRAGMA journal_mode=WAL;'")

In [ ]:
# Cell 3a — Helpers: parallel phase launcher + job-pool runner

def _make_worker_env(gpu_id):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    return env


def run_phase_parallel(phase, n_total_trials, n_gpus):
    """Launch n_gpus workers that share sweep/results/optuna.db via Optuna storage.
    Each asks for ceil(n_total / n_gpus) trials; Optuna coordinates via the DB."""
    trials_per_worker = max(1, math.ceil(n_total_trials / n_gpus))
    print(f"[Phase {phase}] launching {n_gpus} workers × {trials_per_worker} trials")
    procs = []
    for i in range(n_gpus):
        log_path = LOG_DIR / f"phase{phase}_gpu{i}.log"
        log_f = open(log_path, "w")
        cmd = [
            sys.executable, "-m", "sweep.run_sweep",
            "--phase", str(phase),
            "--n-trials", str(trials_per_worker),
        ]
        p = subprocess.Popen(
            cmd, env=_make_worker_env(i),
            stdout=log_f, stderr=subprocess.STDOUT, cwd=str(REPO_ROOT),
        )
        procs.append((p, log_f, log_path, i))
        print(f"  GPU {i}: PID {p.pid} → {log_path}")
    for p, log_f, log_path, i in procs:
        p.wait()
        log_f.close()
        status = "ok" if p.returncode == 0 else f"FAILED (rc={p.returncode})"
        print(f"  GPU {i}: {status}  (tail of {log_path.name}): see file")
    print(f"[Phase {phase}] all workers complete")


def run_jobs_parallel(jobs, n_gpus, label="jobs"):
    """Run a list of independent (config, train, val, test) jobs across GPUs.
    Each job spawns scripts/run_fold.py in its own process. Returns metrics in
    the same order as jobs."""
    gpu_q = queue.Queue()
    for i in range(n_gpus):
        gpu_q.put(i)

    def _run_one(job):
        gpu_id = gpu_q.get()
        cfg_f = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
        json.dump(job["config"], cfg_f); cfg_f.close()
        out_path = Path(tempfile.gettempdir()) / f"metrics_{job['ckpt_suffix']}.json"
        log_path = LOG_DIR / f"fold_{job['ckpt_suffix']}.log"
        try:
            cmd = [
                sys.executable, str(REPO_ROOT / "scripts" / "run_fold.py"),
                "--config-json", cfg_f.name,
                "--train-agents", ",".join(job["train_agents"]),
                "--val-agents", ",".join(job["val_agents"]),
                "--test-agents", ",".join(job["test_agents"]),
                "--out-json", str(out_path),
                "--checkpoint-suffix", job["ckpt_suffix"],
            ]
            with open(log_path, "w") as lf:
                print(f"  [GPU {gpu_id}] {job['label']} starting → {log_path.name}")
                result = subprocess.run(
                    cmd, env=_make_worker_env(gpu_id),
                    stdout=lf, stderr=subprocess.STDOUT, cwd=str(REPO_ROOT),
                )
            if result.returncode != 0:
                raise RuntimeError(f"{job['label']} failed (rc={result.returncode}) — see {log_path}")
            with open(out_path) as f:
                metrics = json.load(f)
            print(f"  [GPU {gpu_id}] {job['label']} F1={metrics['f1']:.4f} AUROC={metrics['auroc']:.4f}")
            return metrics
        finally:
            gpu_q.put(gpu_id)
            Path(cfg_f.name).unlink(missing_ok=True)
            Path(out_path).unlink(missing_ok=True)

    print(f"[{label}] dispatching {len(jobs)} jobs across {n_gpus} GPUs")
    results = [None] * len(jobs)
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_gpus) as ex:
        futures = {ex.submit(_run_one, j): i for i, j in enumerate(jobs)}
        for fut in concurrent.futures.as_completed(futures):
            results[futures[fut]] = fut.result()
    print(f"[{label}] complete")
    return results

In [ ]:
# Cell 3b — Run Phases 1–3 in parallel (gated by RUN_SWEEP)
if RUN_SWEEP:
    for phase in (1, 2, 3):
        run_phase_parallel(phase, PHASE_TRIALS[phase], N_GPUS)
else:
    print("RUN_SWEEP=False — skipping Phases 1–3.")

In [ ]:
# Cell 3c — Phase 4 parallel: 25 (top-K × folds) jobs across GPUs

def run_phase4_parallel(base_cfg, top_k, n_folds, n_gpus):
    import optuna
    storage = os.environ.get("OPTUNA_STORAGE") or f"sqlite:///{SWEEP_RESULTS / 'optuna.db'}"
    study3 = optuna.load_study(study_name="phase3_loss_data", storage=storage)
    top_trials = sorted(study3.trials, key=lambda t: t.value or 0, reverse=True)[:top_k]
    print(f"[Phase 4] loaded {len(top_trials)} top trials from phase3")

    dcfg = base_cfg["data"]
    folds = make_stratified_folds(dcfg["attacked_agents"], dcfg["control_agents"], n_folds)

    jobs = []
    for rank, trial in enumerate(top_trials):
        # Mirrors original run_phase4 semantics: override only with this trial's params.
        cfg = override_config(base_cfg, trial.params)
        for fold_idx in range(n_folds):
            test_agents = folds[fold_idx]
            val_agents = folds[(fold_idx + 1) % n_folds]
            train_agents = [a for j in range(n_folds) for a in folds[j]
                            if j != fold_idx and j != (fold_idx + 1) % n_folds]
            jobs.append({
                "config": cfg,
                "train_agents": train_agents,
                "val_agents": val_agents,
                "test_agents": test_agents,
                "ckpt_suffix": f"phase4_r{rank}_f{fold_idx}",
                "label": f"phase4 rank{rank} fold{fold_idx} (trial {trial.number})",
                "_rank": rank, "_fold": fold_idx,
                "_trial_number": trial.number, "_trial_value": trial.value,
                "_params": trial.params,
            })
    metrics_list = run_jobs_parallel(jobs, n_gpus, label="Phase 4")

    by_rank = {}
    for job, m in zip(jobs, metrics_list):
        by_rank.setdefault(job["_rank"], []).append((job, m))

    summary = []
    for rank in sorted(by_rank.keys()):
        rows = by_rank[rank]
        aurocs = [m["auroc"] for _, m in rows]
        first = rows[0][0]
        summary.append({
            "rank": rank + 1,
            "phase3_trial": first["_trial_number"],
            "params": first["_params"],
            "fold_aurocs": aurocs,
            "mean_auroc": float(np.mean(aurocs)),
            "std_auroc": float(np.std(aurocs)),
        })
    with open(SWEEP_RESULTS / "phase4_results.json", "w") as f:
        json.dump(summary, f, indent=2)
    best = max(summary, key=lambda r: r["mean_auroc"])
    with open(SWEEP_RESULTS / "best_params_phase4.json", "w") as f:
        json.dump(best["params"], f, indent=2)
    print(f"[Phase 4] best: rank {best['rank']} (trial {best['phase3_trial']}) "
          f"mean AUROC={best['mean_auroc']:.4f} ± {best['std_auroc']:.4f}")
    return summary

if RUN_SWEEP:
    phase4_summary = run_phase4_parallel(base_config, PHASE4_TOP_K, PHASE4_FOLDS, N_GPUS)
else:
    print("RUN_SWEEP=False — skipping Phase 4.")

In [ ]:
# Cell 4 — Load best params and build tuned_config
# best_params_phase{1..4}.json store FLAT keys (e.g. "d_model", "lr"), not
# dotted paths. Scan config[model|training|data] — mirrors utils/apply_best_params.py.

def merge_flat_best_params(base_cfg, flat_params):
    cfg = copy.deepcopy(base_cfg)
    changed, unknown = [], []
    for key, value in flat_params.items():
        placed = False
        for section in ("model", "training", "data"):
            if key in cfg.get(section, {}):
                old = cfg[section][key]
                if old != value:
                    changed.append((section, key, old, value))
                cfg[section][key] = value
                placed = True
                break
        if not placed:
            unknown.append((key, value))
    return cfg, changed, unknown

phase_jsons = [SWEEP_RESULTS / f"best_params_phase{p}.json" for p in (1, 2, 3, 4)]
merged_params = {}
for path in phase_jsons:
    if path.exists():
        with open(path) as f:
            phase_params = json.load(f)
        merged_params.update(phase_params)
        print(f"Loaded {path.name}: {len(phase_params)} params")
    else:
        print(f"MISSING {path.name} — skipping")

if not merged_params:
    raise FileNotFoundError(
        "No best_params_phase*.json under sweep/results/. "
        "Set RUN_SWEEP=True and run Cells 3b–3c, or copy sweep results into place."
    )

tuned_config, changed, unknown = merge_flat_best_params(base_config, merged_params)

print(f"\n{'Section':<10} {'Key':<25} {'Baseline':>15} → {'Tuned':<15}")
print("-" * 72)
for section, key, old, new in changed:
    print(f"{section:<10} {key:<25} {str(old):>15} → {str(new):<15}")
if unknown:
    print(f"\nUnknown keys (ignored): {unknown}")

assert tuned_config != base_config, "tuned_config is identical to base_config — merge did nothing"
print(f"\nMerged {len(changed)} changed params.")

In [ ]:
# Cell 5 — Parallel 5-fold CV runner

def run_5fold_cv_parallel(config, label, n_folds=N_FOLDS, n_gpus=N_GPUS):
    dcfg = config["data"]
    folds = make_stratified_folds(dcfg["attacked_agents"], dcfg["control_agents"], n_folds)

    print(f"\n{'=' * 60}\nCV run: {label}\n{'=' * 60}")
    for i, f in enumerate(folds):
        print(f"  Fold {i+1}: {f}")

    jobs = []
    for fold_idx in range(n_folds):
        test_agents = folds[fold_idx]
        val_agents = folds[(fold_idx + 1) % n_folds]
        train_agents = [a for j in range(n_folds) for a in folds[j]
                        if j != fold_idx and j != (fold_idx + 1) % n_folds]
        jobs.append({
            "config": config,
            "train_agents": train_agents,
            "val_agents": val_agents,
            "test_agents": test_agents,
            "ckpt_suffix": f"nb_{label}_f{fold_idx}",
            "label": f"{label} fold{fold_idx+1}",
        })

    metrics_list = run_jobs_parallel(jobs, n_gpus, label=f"CV:{label}")
    rows = []
    for fold_idx, m in enumerate(metrics_list):
        r = dict(m)
        r["fold"] = fold_idx + 1
        r["variant"] = label
        rows.append(r)
    return rows

In [ ]:
# Cell 6 — Parallel CV on baseline config
baseline_rows = run_5fold_cv_parallel(base_config, "baseline")

In [ ]:
# Cell 7 — Parallel CV on tuned config
tuned_rows = run_5fold_cv_parallel(tuned_config, "tuned")

In [ ]:
# Cell 8 — Comparison table
all_rows = baseline_rows + tuned_rows
per_fold_df = pd.DataFrame([{k: v for k, v in r.items() if k != "confusion_matrix"} for r in all_rows])

metric_cols = ["f1", "auroc", "auprc", "precision", "recall", "accuracy"]
summary = per_fold_df.groupby("variant")[metric_cols].agg(["mean", "std"])

rows = []
for m in metric_cols:
    b_mean = summary.loc["baseline", (m, "mean")]
    b_std = summary.loc["baseline", (m, "std")]
    t_mean = summary.loc["tuned", (m, "mean")]
    t_std = summary.loc["tuned", (m, "std")]
    rows.append({
        "metric": m.upper(),
        "baseline": f"{b_mean:.4f} ± {b_std:.4f}",
        "tuned": f"{t_mean:.4f} ± {t_std:.4f}",
        "delta": f"{t_mean - b_mean:+.4f}",
    })
comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

comparison_df.to_csv(RESULTS_DIR / "sweep_comparison.csv", index=False)
per_fold_df.to_csv(RESULTS_DIR / "sweep_folds.csv", index=False)
print(f"\nSaved: {RESULTS_DIR / 'sweep_comparison.csv'}")
print(f"Saved: {RESULTS_DIR / 'sweep_folds.csv'}")

In [ ]:
# Cell 9 — Confusion matrix grid
fig, axes = plt.subplots(2, N_FOLDS, figsize=(3.2 * N_FOLDS, 6.4))
for row_idx, (label, rows) in enumerate([("baseline", baseline_rows), ("tuned", tuned_rows)]):
    for col_idx, fold_row in enumerate(rows):
        ax = axes[row_idx, col_idx]
        cm = np.array(fold_row["confusion_matrix"])
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["normal", "attack"], yticklabels=["normal", "attack"],
        )
        ax.set_title(f"{label} fold {fold_row['fold']}\nF1={fold_row['f1']:.3f}")
        ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout()
fig_path = RESULTS_DIR / "sweep_confusion_matrices.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# Cell 10 — JSON summary artifact
summary_payload = {
    "n_folds": N_FOLDS,
    "n_gpus": N_GPUS,
    "run_sweep": RUN_SWEEP,
    "changed_params": [
        {"section": s, "key": k, "baseline": str(o), "tuned": str(n)}
        for s, k, o, n in changed
    ],
    "per_fold": all_rows,
    "summary": {
        variant: {
            m: {
                "mean": float(summary.loc[variant, (m, "mean")]),
                "std": float(summary.loc[variant, (m, "std")]),
            }
            for m in metric_cols
        }
        for variant in ("baseline", "tuned")
    },
}
summary_path = RESULTS_DIR / "sweep_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary_payload, f, indent=2)
print(f"Saved: {summary_path}")
print("\nDone. Key artifacts:")
print(f"  - {RESULTS_DIR / 'sweep_comparison.csv'}")
print(f"  - {RESULTS_DIR / 'sweep_folds.csv'}")
print(f"  - {RESULTS_DIR / 'sweep_confusion_matrices.png'}")
print(f"  - {summary_path}")